# IN16: Capstone Problem Statement
## Production AI System Design for Walmart Global Tech

**Programme:** Advanced Agentic AI -- Production Engineering
**Track:** India | Walmart Global Tech Academy
**Module:** 5 -- AI Economics, Optimization and Architecture Review

---

## Business Context

Walmart India is expanding its digital retail assistant capabilities.
The current system handles simple FAQ lookups but cannot handle
multi-step customer queries that require tool use, cross-product comparison,
or order management actions.

You are the lead AI engineer. You have been asked to design, implement, evaluate,
and defend a production-grade Walmart Retail Assistant that will handle
**100,000 customer queries per day** across five query categories.

At the end of this capstone you will present your solution to a simulated
**Architecture Review Board (ARB)** covering all six required components.

---

## The Six ARB Components You Must Deliver

| # | Component | Deliverable |
|---|---|---|
| 1 | Architecture choice | Scored decision matrix + ADR |
| 2 | Agent strategy | Implemented orchestration with tool dispatch |
| 3 | Evaluation strategy | 10-metric scorecard (golden dataset) |
| 4 | Cost model | Monthly projection + optimisation levers |
| 5 | Deployment model | CI/CD plan + rollback procedure |
| 6 | Risk mitigation plan | Security + hallucination + drift controls |

---

## Constraints

- All API keys must be loaded from `.env` using `load_dotenv(override=True)`.
  Never hardcode keys.
- Primary model: `gpt-4o-mini` for cost efficiency.
  Use `gpt-4-turbo` only for complex multi-intent queries.
- Retrieval: Pinecone serverless index `walmart-rag` (dimension=1536, cosine).
  If unavailable, use the mock retriever provided below.
- Token budget per call: max 800 input tokens, 150 output tokens.
- P95 latency SLO: under 3000ms.
- Monthly spend ceiling: $1,500.
- All evaluation thresholds from Module 4 (IN10) must be met.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openai', 'python-dotenv', 'tiktoken'], check=True)
print('Packages ready.')

In [1]:
import os, json, time, uuid, hashlib
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Environment loaded.')

Environment loaded.


## Provided: Mock Knowledge Base and Tool Definitions

The following knowledge base and tool stubs are provided.
Do NOT modify these -- use them as-is in your implementation.

In [2]:
# Mock knowledge base (use this if Pinecone is unavailable)
KNOWLEDGE_BASE = [
    {'id': 'P001', 'category': 'price',  'text': 'Great Value Whole Milk 1 gallon is priced at $3.98. Located in Aisle 12, Dairy section. In stock: 47 units.'},
    {'id': 'P002', 'category': 'price',  'text': 'Great Value 2% Milk 1 gallon is priced at $3.78. Located in Aisle 12, Dairy section. In stock: 32 units.'},
    {'id': 'P003', 'category': 'price',  'text': 'Tide Original Laundry Detergent 92 oz is $11.97 (13 cents/oz). Aisle 7, Cleaning supplies.'},
    {'id': 'P004', 'category': 'price',  'text': 'Great Value Laundry Detergent 150 oz is $8.97 (6 cents/oz). Aisle 7, Cleaning supplies.'},
    {'id': 'O001', 'category': 'order',  'text': 'Order ORD-78901: shipped via FedEx, tracking FX123456, estimated delivery July 3 2026.'},
    {'id': 'O002', 'category': 'order',  'text': 'Order ORD-45621: processing, expected to ship within 2 business days.'},
    {'id': 'R001', 'category': 'policy', 'text': 'Electronics return policy: 30 days with receipt and original packaging. No exceptions.'},
    {'id': 'R002', 'category': 'policy', 'text': 'General return policy: 90 days with or without receipt. Without receipt: valid photo ID required, refund as store credit.'},
    {'id': 'H001', 'category': 'hours',  'text': 'Most Walmart Supercenters open at 6:00 AM and close at 11:00 PM Monday through Saturday.'},
    {'id': 'H002', 'category': 'hours',  'text': 'Sunday hours: 7:00 AM to 10:00 PM. Walmart stores are closed on Thanksgiving Day.'},
]

def mock_retrieve(query: str, k: int = 3) -> list:
    """Simple keyword-based mock retriever."""
    q = query.lower()
    scored = []
    for doc in KNOWLEDGE_BASE:
        score = sum(1 for word in q.split() if word in doc['text'].lower())
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for _, doc in scored[:k]]

# Tool stubs -- these simulate real tool calls
def price_lookup(product_name: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'price' and
               product_name.lower() in d['text'].lower()]
    return {'found': len(results) > 0, 'results': results}

def order_status(order_id: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'order' and
               order_id in d['text']]
    return {'found': len(results) > 0, 'results': results}

def policy_search(topic: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'policy' and
               any(w in d['text'].lower() for w in topic.lower().split())]
    return {'found': len(results) > 0, 'results': results}

def store_hours(day: str = '') -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'hours']
    return {'found': True, 'results': results}

TOOLS = {
    'price_lookup':  price_lookup,
    'order_status':  order_status,
    'policy_search': policy_search,
    'store_hours':   store_hours,
}
print('Knowledge base and tools ready.')
print(f'Documents: {len(KNOWLEDGE_BASE)} | Tools: {list(TOOLS.keys())}')

Knowledge base and tools ready.
Documents: 10 | Tools: ['price_lookup', 'order_status', 'policy_search', 'store_hours']


---
## Task 1: Architecture Choice -- Decision Matrix and ADR

**What to build:** A scored decision matrix comparing three architecture options
for the Walmart Retail Assistant, followed by an ADR documenting your chosen approach.

**Requirements:**
- Evaluate at least three options: RAG + Agent, RAG + Workflow, Traditional Search
- Use the four-axis framework: cost (30%), latency (25%), quality (30%), maintainability (15%)
- Score each option 1-5 per axis
- Write an ADR for the winning option
- Save the ADR to `capstone_adr.txt`

**ARB question you must be able to answer:**
> 'Why an agent and not a deterministic workflow for this use case?'

In [3]:
# ── TODO 1: Build the decision matrix ────────────────────────────────────
# Define your options, criteria with weights, and scores.
# Use the decision_matrix() pattern from IN15.

options  = [
    # TODO: list your three architecture options
]

criteria = [
    # TODO: (criterion_name, weight) -- weights must sum to 1.0
]

scores = {
    # TODO: {option: {criterion: score_1_to_5}}
}

# TODO: Print the decision matrix and identify the winner


# ── TODO 2: Write the ADR ─────────────────────────────────────────────────
# Use the generate_adr() pattern from IN15.
# Save to capstone_adr.txt

adr_text = ''  # TODO: generate ADR
Path('capstone_adr.txt').write_text(adr_text)
print('ADR saved.')

ADR saved.


---
## Task 2: Agent Strategy -- Orchestration and Tool Dispatch

**What to build:** A `WalmartRetailAgent` class that:
- Classifies the incoming query into one of five categories
  (price, order, policy, hours, multi_step)
- Routes the query to the appropriate tool
- Retrieves context using `mock_retrieve()`
- Calls the LLM with the retrieved context
- Returns a structured response with trace metadata

**Requirements:**
- Use `gpt-4o-mini` for single-category queries
- Use `gpt-4-turbo` for `multi_step` queries
- Every response must include: answer, model_used, input_tokens,
  output_tokens, cost_usd, latency_ms, tool_used
- Input tokens must not exceed 800; output tokens must not exceed 150

**ARB question you must be able to answer:**
> 'What happens when the query classifier misclassifies a query?'

In [4]:
MODEL_PRICING = {
    'gpt-4-turbo': {'input': 10.00, 'output': 30.00},
    'gpt-4o':      {'input':  5.00, 'output': 15.00},
    'gpt-4o-mini': {'input':  0.15, 'output':  0.60},
}

def compute_cost(model, in_tok, out_tok):
    p = MODEL_PRICING[model]
    return round((in_tok/1_000_000)*p['input'] + (out_tok/1_000_000)*p['output'], 6)


class WalmartRetailAgent:
    SYSTEM_PROMPT = (
        'You are the Walmart Retail Assistant. '
        'Answer the customer query using only the provided context. '
        'Be concise and accurate. If the answer is not in the context, say so.'
    )

    def classify(self, query: str) -> str:
        # TODO: Classify query into: price | order | policy | hours | multi_step
        # Use keyword matching or LLM classification
        pass

    def select_tool(self, category: str, query: str) -> dict:
        # TODO: Select and call the appropriate tool from TOOLS dict
        # Return {'tool_used': name, 'result': tool_output}
        pass

    def select_model(self, category: str) -> str:
        # TODO: Return 'gpt-4o-mini' for simple queries,
        #       'gpt-4-turbo' for multi_step
        pass

    def run(self, query: str) -> dict:
        # TODO: Orchestrate the full pipeline:
        # 1. Classify query
        # 2. Select and call tool
        # 3. Retrieve context with mock_retrieve()
        # 4. Call LLM with context (max 800 input, 150 output tokens)
        # 5. Return structured response
        pass


# ── Test your agent on 5 queries ──────────────────────────────────────────
agent = WalmartRetailAgent()
test_queries = [
    'What is the price of Great Value Whole Milk?',
    'Where is my order ORD-78901?',
    'What is the return policy for electronics?',
    'What time does Walmart open on Sunday?',
    'Compare Great Value and Tide detergent on price per ounce and tell me which is cheaper.',
]
# TODO: Run agent on each query and print results

---
## Task 3: Evaluation Strategy -- 10-Metric Scorecard

**What to build:** Run your agent against the golden dataset and produce a
pass/fail scorecard across all 10 evaluation metrics.

**Golden dataset (10 records -- use these exactly):**

In [5]:
GOLDEN_DATASET = [
    {'id':'G001','cat':'price',
     'query':'What is the price of Great Value Whole Milk?',
     'expected':'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.'},
    {'id':'G002','cat':'price',
     'query':'Which laundry detergent costs less per ounce?',
     'expected':'Great Value at 6 cents/oz is cheaper than Tide at 13 cents/oz.'},
    {'id':'G003','cat':'order',
     'query':'What is the status of order ORD-78901?',
     'expected':'Order ORD-78901 has been shipped via FedEx (FX123456), arriving by July 3, 2026.'},
    {'id':'G004','cat':'order',
     'query':'When will order ORD-45621 ship?',
     'expected':'Order ORD-45621 is being processed and will ship within 2 business days.'},
    {'id':'G005','cat':'policy',
     'query':'Can I return electronics after 30 days?',
     'expected':'No. Electronics must be returned within 30 days with receipt and original packaging.'},
    {'id':'G006','cat':'policy',
     'query':'Can I return an item without a receipt?',
     'expected':'Yes, within 90 days with a valid photo ID. Refund is issued as store credit.'},
    {'id':'G007','cat':'hours',
     'query':'What time does Walmart open on weekdays?',
     'expected':'Most Walmart Supercenters open at 6:00 AM Monday through Saturday.'},
    {'id':'G008','cat':'hours',
     'query':'Is Walmart open on Thanksgiving?',
     'expected':'No, Walmart stores are closed on Thanksgiving Day.'},
    {'id':'G009','cat':'multi_step',
     'query':'Is Great Value Whole Milk in stock and what aisle?',
     'expected':'Great Value Whole Milk is in stock with 47 units in Aisle 12, Dairy section.'},
    {'id':'G010','cat':'multi_step',
     'query':'Compare Great Value and Tide detergent on price per ounce.',
     'expected':'Great Value (150 oz) costs $8.97 at 6c/oz. Tide (92 oz) costs $11.97 at 13c/oz. Great Value is cheaper per ounce.'},
]

# ── TODO 3: Evaluate your agent on the golden dataset ─────────────────────
# For each record:
# 1. Run agent.run(query)
# 2. Score using LLM-as-judge (0-3 rubric from IN11, normalised /3)
# 3. Compute: faithfulness, answer_relevancy, hit_rate, task_success_rate
# 4. Generate pass/fail per metric using thresholds from IN10
# 5. Save scorecard to capstone_evaluation_scorecard.txt

# Thresholds (from IN10):
THRESHOLDS = {
    'faithfulness':       0.85,
    'answer_relevancy':   0.75,
    'toxicity':           0.10,  # must be BELOW this
    'hit_rate_at_3':      0.75,
    'task_success_rate':  0.90,
    'tool_call_accuracy': 0.95,
}

# TODO: Run evaluation and print scorecard

---
## Task 4: Cost Model -- Monthly Projection

**What to build:** A monthly spend projection for your agent at 100,000 calls/day,
with and without three optimisation techniques.

**Requirements:**
- Compute the unoptimised baseline spend (all gpt-4-turbo, no caching)
- Apply model routing (from Task 2 query classification)
- Apply response caching (assume 20% hit rate for retail queries)
- Apply prompt compression (assume 18% token reduction)
- Show monthly spend for each scenario
- Confirm the optimised scenario is within the $1,500/month ceiling

**ARB question you must be able to answer:**
> 'What is your worst-case monthly spend if the cache fails?'

In [6]:
# ── TODO 4: Monthly spend projection ─────────────────────────────────────
# Use the monthly_projection() pattern from IN13.

DAILY_CALLS = 100_000

# Measure average tokens from your Task 2 agent runs
avg_input_tokens  = 0  # TODO: compute from your agent test runs
avg_output_tokens = 0  # TODO: compute from your agent test runs

# TODO: Compute three scenarios and print comparison table
# Scenario 1: Baseline (all gpt-4-turbo, no optimisation)
# Scenario 2: Model routing from Task 2
# Scenario 3: Scenario 2 + 20% cache + 18% compression

# TODO: Confirm Scenario 3 is within $1,500/month ceiling
MONTHLY_CEILING = 1500.00
# assert optimised_monthly <= MONTHLY_CEILING, 'Budget ceiling breached'

---
## Task 5: Deployment Model -- CI/CD and Rollback

**What to build:** A written deployment model document covering:
- The CI/CD pipeline for model and prompt changes
- The rollback procedure (target: under 30 minutes)
- The on-call escalation path
- Save to `capstone_deployment_model.txt`

**Required sections:**
1. Change types and their pipeline paths
2. Pre-deployment gate (evaluation scorecard must PASS)
3. Deployment steps (staging -> canary -> production)
4. Rollback trigger conditions
5. Rollback steps
6. On-call runbook summary

**ARB question you must be able to answer:**
> 'If a prompt change degrades faithfulness score from 0.88 to 0.76, how quickly can you roll back?'

In [7]:
# ── TODO 5: Write the deployment model document ───────────────────────────

deployment_model = '''
CAPSTONE DEPLOYMENT MODEL -- Walmart Retail Assistant
=====================================================

1. CHANGE TYPES AND PIPELINE PATHS
   # TODO: Define pipeline for each change type:
   # - System prompt change
   # - Model version change
   # - RAG chunk configuration change
   # - Tool logic change

2. PRE-DEPLOYMENT GATE
   # TODO: List the evaluation gates that must PASS before deploy

3. DEPLOYMENT STEPS
   # TODO: staging -> canary -> production with % traffic

4. ROLLBACK TRIGGER CONDITIONS
   # TODO: List conditions that trigger immediate rollback

5. ROLLBACK STEPS
   # TODO: Step-by-step rollback procedure (target < 30 min)

6. ON-CALL RUNBOOK SUMMARY
   # TODO: Who to page, what to check, first 5 actions
'''

Path('capstone_deployment_model.txt').write_text(deployment_model)
print('Deployment model saved.')

Deployment model saved.


---
## Task 6: Risk Mitigation Plan

**What to build:** A structured risk register with mitigations for six risk categories.
Save to `capstone_risk_register.txt`.

**Required risk categories:**
1. Hallucination (answer not grounded in context)
2. Prompt injection (user attempts to override system prompt)
3. PII leakage (order numbers, personal data in logs)
4. Model drift (quality degradation over time)
5. Cost overrun (token budget exceeded)
6. Dependency failure (OpenAI API, Pinecone unavailable)

**For each risk, document:** Likelihood, Impact, Detection method, Mitigation, Owner

**ARB question you must be able to answer:**
> 'What is your detection and response plan for silent quality drift?'

In [8]:
# ── TODO 6: Risk register ──────────────────────────────────────────────────

RISK_CATEGORIES = [
    'hallucination',
    'prompt_injection',
    'pii_leakage',
    'model_drift',
    'cost_overrun',
    'dependency_failure',
]

# TODO: For each risk category, fill in the register
risk_register = {
    cat: {
        'likelihood':  '',  # Low / Medium / High
        'impact':      '',  # Low / Medium / High / Critical
        'detection':   '',  # How do you know it is happening?
        'mitigation':  '',  # What do you do about it?
        'owner':       '',  # Team responsible
    }
    for cat in RISK_CATEGORIES
}

# TODO: Print and save the risk register
Path('capstone_risk_register.txt').write_text(json.dumps(risk_register, indent=2))
print('Risk register saved.')

Risk register saved.


---
## Deliverables Checklist

Before submitting your solution (IN17), verify all files have been generated:

| File | Task | Required |
|---|---|---|
| `capstone_adr.txt` | Task 1 | Architecture Decision Record |
| `capstone_evaluation_scorecard.txt` | Task 3 | All 10 metrics with pass/fail |
| `capstone_deployment_model.txt` | Task 5 | CI/CD and rollback plan |
| `capstone_risk_register.txt` | Task 6 | Six risk categories with mitigations |

**ARB Presentation Requirement:**
Prepare a 6-section verbal defence of your solution covering all components above.
Peer reviewers will ask questions from the 20-question ARB list in IN15.

---